# Real ONT Benchmark Driver

This notebook is a benchmark-grade experiment driver for OMEGA on GIAB-style ONT data.

It now supports:
- phased VCF and haplotagged BAM plumbing through preprocessing
- region BED annotations for homopolymers, repeats, segmental duplications, low-complexity sequence, and variant-rich regions
- HG002 primary training plus HG003/HG004 transfer evaluation planning
- executable hooks for internal baselines, external baselines, corrected-read export, assembly, and variant-calling evaluation
- environment-variable or workspace-relative path resolution instead of hardcoded local paths
- device fallback (`cuda -> mps -> cpu`) when configs are generated
- fail-fast validation before execution when required files are missing

Recommended environment variables for the trio preset:
- `HG002_READS_BAM`, `HG002_HAPLOTAGGED_BAM`, `HG002_PHASED_VCF`
- `HG003_READS_BAM`, `HG003_HAPLOTAGGED_BAM`, `HG003_TRUTH_VCF`, `HG003_CONFIDENT_BED`, `HG003_PHASED_VCF`
- `HG004_READS_BAM`, `HG004_HAPLOTAGGED_BAM`, `HG004_TRUTH_VCF`, `HG004_CONFIDENT_BED`, `HG004_PHASED_VCF`
- `GIAB_HOMOPOLYMER_BED`, `GIAB_LOW_COMPLEXITY_BED`, `GIAB_SEGDUP_BED`, `GIAB_REPEAT_BED`, `GIAB_VARIANT_RICH_BED`
- `OMEGA_HERRO_CMD_TEMPLATE`, `OMEGA_DECHAT_CMD_TEMPLATE`
- `OMEGA_ASSEMBLER_CMD_TEMPLATE`, `OMEGA_VARIANT_CALL_CMD_TEMPLATE`, `OMEGA_HAPPY_CMD_TEMPLATE`

The notebook defaults to an immediately runnable local training preset. Use `OMEGA_EXPERIMENT_PRESET=giab_trio_benchmark` once the full GIAB trio inputs are available.

In [6]:
from __future__ import annotations

import copy
import gzip
import hashlib
import importlib
import inspect
import json
import os
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
import torch
import yaml
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'scripts').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / 'src'))

EXPERIMENT_PRESET = os.environ.get('OMEGA_EXPERIMENT_PRESET', 'hg002_chr20_small_benchmark')
SELECTED_RUNS = [token.strip() for token in os.environ.get('OMEGA_SELECTED_RUNS', 'full').split(',') if token.strip()]
PREFERRED_DEVICE = os.environ.get('OMEGA_DEVICE', 'auto')
VALIDATE_ON_PLAN = os.environ.get('OMEGA_VALIDATE_ON_PLAN', '0') == '1'

RUN_PREPROCESS = os.environ.get('OMEGA_RUN_PREPROCESS', '0') == '1'
RUN_TRAIN = os.environ.get('OMEGA_RUN_TRAIN', '1') == '1'
RUN_TEST = os.environ.get('OMEGA_RUN_TEST', '1') == '1'
RUN_BASELINES = os.environ.get('OMEGA_RUN_BASELINES', '0') == '1'
RUN_EXTERNAL_BASELINES = os.environ.get('OMEGA_RUN_EXTERNAL_BASELINES', '0') == '1'
RUN_DOWNSTREAM = os.environ.get('OMEGA_RUN_DOWNSTREAM', '0') == '1'

PROFILE_PRESETS = {
    'giab_hg002_local': {
        'mode': 'prealigned_benchmark',
        'run_name': 'giab_hg002_local',
        'datasets': [
            {
                'name': 'hg002_local',
                'role': 'primary',
                'reference_fasta': {'default': 'data/raw/giab/hg002/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna', 'required': True},
                'reads_bam': {'env': 'HG002_READS_BAM', 'default': 'data/raw/giab/hg002/HG002_GRCh38_ONT-UL_UCSC_20200508.phased.bam', 'required': True},
                'truth_vcf': {'default': 'data/raw/giab/hg002/HG002_GRCh38_1_22_v4.2.1_benchmark.vcf.gz', 'required': True},
                'confident_bed': {'default': 'data/raw/giab/hg002/HG002_GRCh38_1_22_v4.2.1_benchmark_noinconsistent.bed', 'required': True},
                'phased_vcf': {'env': 'HG002_PHASED_VCF', 'default': '', 'required': False},
                'haplotagged_bam': {'env': 'HG002_HAPLOTAGGED_BAM', 'default': '', 'required': False},
                'train_contigs': 'chr1,chr2,chr3,chr4,chr5,chr6,chr7,chr8,chr9,chr10,chr11,chr12,chr13,chr14,chr15,chr16',
                'val_contigs': 'chr17,chr18,chr19',
                'test_contigs': 'chr20,chr21,chr22,chrX',
            },
        ],
        'region_beds': {
            'homopolymer': {'env': 'GIAB_HOMOPOLYMER_BED', 'default': '', 'required': False},
            'low_complexity': {'env': 'GIAB_LOW_COMPLEXITY_BED', 'default': '', 'required': False},
            'segmental_duplication': {'env': 'GIAB_SEGDUP_BED', 'default': '', 'required': False},
            'repeat': {'env': 'GIAB_REPEAT_BED', 'default': '', 'required': False},
            'variant_rich': {'env': 'GIAB_VARIANT_RICH_BED', 'default': '', 'required': False},
        },
        'checkpoint_weights': {'overcorrection': 3.0, 'hard_edit_fp': 2.0, 'length_ratio': 0.25},
        'stages': [
            {'name': 'stage1_1kb', 'window_size': 1024, 'window_overlap': 256, 'min_window_size': 512, 'max_target_len': 1024, 'max_support_len': 1024, 'batch_size': 4, 'epochs': 8, 'min_read_length': 10000, 'max_supports': 16, 'min_supports_per_window': 4, 'min_mapq': 20},
            {'name': 'stage2_4kb', 'window_size': 4096, 'window_overlap': 1024, 'min_window_size': 1024, 'max_target_len': 4096, 'max_support_len': 4096, 'batch_size': 2, 'epochs': 8, 'min_read_length': 10000, 'max_supports': 24, 'min_supports_per_window': 6, 'min_mapq': 20},
            {'name': 'stage3_8kb', 'window_size': 8192, 'window_overlap': 2048, 'min_window_size': 2048, 'max_target_len': 8192, 'max_support_len': 8192, 'batch_size': 1, 'epochs': 6, 'min_read_length': 10000, 'max_supports': 32, 'min_supports_per_window': 8, 'min_mapq': 20},
        ],
    },
    'local_ready_training': {
        'mode': 'preprocessed_dataset',
        'run_name': 'local_ready_training',
        'datasets': [
            {
                'name': 'uti89_local',
                'role': 'primary',
                'manifest_path': {'default': 'data/real_uti89_notebook/manifest.json', 'required': True},
                'train_path': {'default': 'data/real_uti89_notebook/train.jsonl', 'required': True},
                'val_path': {'default': 'data/real_uti89_notebook/val.jsonl', 'required': True},
                'test_path': {'default': 'data/real_uti89_notebook/test.jsonl', 'required': True},
                'reference_fasta': {'default': 'data/raw/uti89/reference.fna', 'required': False},
                'reads_bam': {'default': 'data/real_uti89_notebook/aligned.sorted.bam', 'required': False},
                'truth_vcf': {'default': '', 'required': False},
                'confident_bed': {'default': '', 'required': False},
                'phased_vcf': {'default': '', 'required': False},
                'haplotagged_bam': {'default': 'data/real_uti89_notebook/aligned.sorted.bam', 'required': False},
                'train_contigs': '',
                'val_contigs': '',
                'test_contigs': '',
            },
        ],
        'region_beds': {},
        'checkpoint_weights': {'overcorrection': 3.0, 'hard_edit_fp': 2.0, 'length_ratio': 0.25},
        'stages': [
            {'name': 'stage1_local', 'window_size': 1024, 'window_overlap': 256, 'min_window_size': 512, 'max_target_len': 1024, 'max_support_len': 1024, 'batch_size': 4, 'epochs': 10, 'min_read_length': 0, 'max_supports': 16, 'min_supports_per_window': 0, 'min_mapq': 0},
        ],
    },
    'giab_trio_benchmark': {
        'mode': 'prealigned_benchmark',
        'run_name': 'giab_trio_benchmark',
        'datasets': [
            {
                'name': 'hg002_primary',
                'role': 'primary',
                'reference_fasta': {'env': 'GIAB_REFERENCE_FASTA', 'default': 'data/raw/giab/hg002/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna', 'required': True},
                'reads_bam': {'env': 'HG002_READS_BAM', 'default': '', 'required': True},
                'truth_vcf': {'env': 'HG002_TRUTH_VCF', 'default': 'data/raw/giab/hg002/HG002_GRCh38_1_22_v4.2.1_benchmark.vcf.gz', 'required': True},
                'confident_bed': {'env': 'HG002_CONFIDENT_BED', 'default': 'data/raw/giab/hg002/HG002_GRCh38_1_22_v4.2.1_benchmark_noinconsistent.bed', 'required': True},
                'phased_vcf': {'env': 'HG002_PHASED_VCF', 'default': '', 'required': False},
                'haplotagged_bam': {'env': 'HG002_HAPLOTAGGED_BAM', 'default': '', 'required': False},
                'train_contigs': 'chr1,chr2,chr3,chr4,chr5,chr6,chr7,chr8,chr9,chr10,chr11,chr12,chr13,chr14,chr15,chr16',
                'val_contigs': 'chr17,chr18,chr19',
                'test_contigs': 'chr20,chr21,chr22,chrX',
            },
            {
                'name': 'hg003_transfer',
                'role': 'eval_only',
                'reference_fasta': {'env': 'GIAB_REFERENCE_FASTA', 'default': 'data/raw/giab/hg002/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna', 'required': True},
                'reads_bam': {'env': 'HG003_READS_BAM', 'default': '', 'required': True},
                'truth_vcf': {'env': 'HG003_TRUTH_VCF', 'default': '', 'required': True},
                'confident_bed': {'env': 'HG003_CONFIDENT_BED', 'default': '', 'required': True},
                'phased_vcf': {'env': 'HG003_PHASED_VCF', 'default': '', 'required': False},
                'haplotagged_bam': {'env': 'HG003_HAPLOTAGGED_BAM', 'default': '', 'required': False},
                'train_contigs': '',
                'val_contigs': '',
                'test_contigs': 'chr20,chr21,chr22,chrX',
            },
            {
                'name': 'hg004_transfer',
                'role': 'eval_only',
                'reference_fasta': {'env': 'GIAB_REFERENCE_FASTA', 'default': 'data/raw/giab/hg002/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna', 'required': True},
                'reads_bam': {'env': 'HG004_READS_BAM', 'default': '', 'required': True},
                'truth_vcf': {'env': 'HG004_TRUTH_VCF', 'default': '', 'required': True},
                'confident_bed': {'env': 'HG004_CONFIDENT_BED', 'default': '', 'required': True},
                'phased_vcf': {'env': 'HG004_PHASED_VCF', 'default': '', 'required': False},
                'haplotagged_bam': {'env': 'HG004_HAPLOTAGGED_BAM', 'default': '', 'required': False},
                'train_contigs': '',
                'val_contigs': '',
                'test_contigs': 'chr20,chr21,chr22,chrX',
            },
        ],
        'region_beds': {
            'homopolymer': {'env': 'GIAB_HOMOPOLYMER_BED', 'default': '', 'required': False},
            'low_complexity': {'env': 'GIAB_LOW_COMPLEXITY_BED', 'default': '', 'required': False},
            'segmental_duplication': {'env': 'GIAB_SEGDUP_BED', 'default': '', 'required': False},
            'repeat': {'env': 'GIAB_REPEAT_BED', 'default': '', 'required': False},
            'variant_rich': {'env': 'GIAB_VARIANT_RICH_BED', 'default': '', 'required': False},
        },
        'checkpoint_weights': {'overcorrection': 3.0, 'hard_edit_fp': 2.0, 'length_ratio': 0.25},
        'stages': [
            {'name': 'stage1_1kb', 'window_size': 1024, 'window_overlap': 256, 'min_window_size': 512, 'max_target_len': 1024, 'max_support_len': 1024, 'batch_size': 4, 'epochs': 8, 'min_read_length': 10000, 'max_supports': 16, 'min_supports_per_window': 4, 'min_mapq': 20},
            {'name': 'stage2_4kb', 'window_size': 4096, 'window_overlap': 1024, 'min_window_size': 1024, 'max_target_len': 4096, 'max_support_len': 4096, 'batch_size': 2, 'epochs': 8, 'min_read_length': 10000, 'max_supports': 24, 'min_supports_per_window': 6, 'min_mapq': 20},
            {'name': 'stage3_8kb', 'window_size': 8192, 'window_overlap': 2048, 'min_window_size': 2048, 'max_target_len': 8192, 'max_support_len': 8192, 'batch_size': 1, 'epochs': 6, 'min_read_length': 10000, 'max_supports': 32, 'min_supports_per_window': 8, 'min_mapq': 20},
        ],
    },
}
from omega_longread.benchmark_presets import NOTEBOOK_PRESETS
PROFILE_PRESETS.update(copy.deepcopy(NOTEBOOK_PRESETS))

ALL_RUNS = [
    {'name': 'full', 'support_mode': 'full', 'description': 'Target + overlap-conditioned support encoder'},
    {'name': 'target_only', 'support_mode': 'target_only', 'description': 'Target encoder only; support path disabled'},
    {'name': 'masked_target', 'support_mode': 'masked_target', 'description': 'Support-conditioned with masked target features'},
]

In [7]:
def resolve_spec(spec, base_dir: Path) -> tuple[str, str | None, bool]:
    if isinstance(spec, str):
        raw = spec.strip()
        env_var = None
        required = bool(raw)
    else:
        env_var = spec.get('env')
        raw = os.environ.get(env_var, '').strip() if env_var else ''
        if not raw:
            raw = str(spec.get('default', '')).strip()
        required = bool(spec.get('required', False))
    if not raw:
        return '', env_var, required
    raw = os.path.expanduser(os.path.expandvars(raw))
    path = Path(raw)
    if not path.is_absolute():
        path = base_dir / path
    return str(path.resolve()), env_var, required


def resolve_dataset_entry(entry: dict, base_dir: Path) -> dict:
    resolved = dict(entry)
    required_fields = {}
    for field in ['manifest_path', 'train_path', 'val_path', 'test_path', 'reference_fasta', 'reads_bam', 'truth_vcf', 'confident_bed', 'phased_vcf', 'haplotagged_bam']:
        value, env_var, required = resolve_spec(entry.get(field, ''), base_dir)
        resolved[field] = value
        required_fields[field] = {'env_var': env_var, 'required': required}
    if not resolved['haplotagged_bam']:
        resolved['haplotagged_bam'] = resolved['reads_bam']
    resolved['_requirements'] = required_fields
    return resolved


def resolve_profile(profile: dict, base_dir: Path) -> dict:
    resolved = copy.deepcopy(profile)
    resolved['datasets'] = [resolve_dataset_entry(dataset, base_dir) for dataset in profile['datasets']]
    region_beds = {}
    region_requirements = {}
    for name, spec in profile.get('region_beds', {}).items():
        path, env_var, required = resolve_spec(spec, base_dir)
        region_beds[name] = path
        region_requirements[name] = {'env_var': env_var, 'required': required}
    resolved['region_beds'] = region_beds
    resolved['_region_requirements'] = region_requirements
    return resolved


def validate_profile_inputs(profile: dict) -> list[dict]:
    rows = []
    for dataset in profile['datasets']:
        for field in ['manifest_path', 'train_path', 'val_path', 'test_path', 'reference_fasta', 'reads_bam', 'truth_vcf', 'confident_bed', 'phased_vcf', 'haplotagged_bam']:
            value = dataset.get(field, '')
            requirement = dataset.get('_requirements', {}).get(field, {})
            required = bool(requirement.get('required', False))
            env_var = requirement.get('env_var')
            rows.append({
                'dataset': dataset['name'],
                'field': field,
                'path': value,
                'exists': bool(value) and Path(value).exists(),
                'required': required,
                'env_var': env_var or '',
            })
    for name, path in profile.get('region_beds', {}).items():
        req = profile.get('_region_requirements', {}).get(name, {})
        rows.append({
            'dataset': '*regions*',
            'field': name,
            'path': path,
            'exists': bool(path) and Path(path).exists(),
            'required': bool(req.get('required', False)),
            'env_var': req.get('env_var') or '',
        })
    return rows


def missing_required_inputs(rows: list[dict]) -> list[dict]:
    return [row for row in rows if row['required'] and not row['exists']]


def pick_device(preferred: str) -> tuple[str, bool]:
    preferred = preferred.lower().strip()
    if preferred in {'', 'auto'}:
        preferred = 'cuda'
    if preferred == 'cuda':
        if torch.cuda.is_available():
            return 'cuda', True
        if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
            return 'mps', False
        return 'cpu', False
    if preferred == 'mps':
        if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
            return 'mps', False
        if torch.cuda.is_available():
            return 'cuda', True
        return 'cpu', False
    return 'cpu', False


def run_cmd(cmd, cwd=ROOT, env=None, dry_run=False):
    pretty = ' '.join(str(x) for x in cmd)
    print(pretty)
    if dry_run:
        return None
    return subprocess.run(cmd, cwd=cwd, env=env, check=True)


def stringify_cmd(cmd):
    return ' '.join(str(x) for x in cmd)


def safe_rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


def ensure_demo_inputs(profile):
    import pysam

    raw_dir = ROOT / 'data' / 'raw' / 'uti89'
    raw_dir.mkdir(parents=True, exist_ok=True)
    reference_gz = raw_dir / 'reference.fna.gz'
    reference_fa = raw_dir / 'reference.fna'
    fastq_urls = profile.get('fastq_urls', [])
    fastq_paths = [raw_dir / Path(url).name for url in fastq_urls]
    downloads = {reference_gz: profile['reference_gz_url']}
    downloads.update({path: url for path, url in zip(fastq_paths, fastq_urls)})
    for path, url in downloads.items():
        if path.exists() and path.stat().st_size > 0:
            continue
        urlretrieve(url, path)
    if not reference_fa.exists():
        with gzip.open(reference_gz, 'rb') as src, open(reference_fa, 'wb') as dst:
            dst.write(src.read())
    if not (reference_fa.with_suffix(reference_fa.suffix + '.fai')).exists():
        pysam.faidx(str(reference_fa))
    return {'reference_fasta': reference_fa, 'fastq_paths': fastq_paths}


def prepare_demo_stage(profile, stage_cfg, stage_dir):
    import mappy as mp
    import pysam
    import omega_longread.preprocessing as preprocessing

    preprocessing = importlib.reload(preprocessing)
    inputs = ensure_demo_inputs(profile)
    reference_fa = Path(inputs['reference_fasta'])
    fastq_paths = [Path(p) for p in inputs['fastq_paths']]
    unsorted_bam = stage_dir / 'aligned.unsorted.bam'
    sorted_bam = stage_dir / 'aligned.sorted.bam'
    train_jsonl = stage_dir / 'train.jsonl'
    val_jsonl = stage_dir / 'val.jsonl'
    test_jsonl = stage_dir / 'test.jsonl'

    def revcomp(seq: str) -> str:
        table = str.maketrans('ACGTNacgtn', 'TGCANtgcan')
        return seq.translate(table)[::-1]

    def iter_fastq(path: Path):
        with gzip.open(path, 'rt') as handle:
            while True:
                name = handle.readline().strip()
                if not name:
                    return
                seq = handle.readline().strip()
                handle.readline()
                qual = handle.readline().strip()
                yield name[1:], seq, qual

    def full_cigar(hit, query_len: int) -> str:
        if hit.strand < 0:
            left_soft = query_len - hit.q_en
            right_soft = hit.q_st
        else:
            left_soft = hit.q_st
            right_soft = query_len - hit.q_en
        parts = []
        if left_soft > 0:
            parts.append(f'{left_soft}S')
        parts.append(hit.cigar_str)
        if right_soft > 0:
            parts.append(f'{right_soft}S')
        return ''.join(parts)

    reads = []
    for fq in fastq_paths:
        for name, seq, qual in iter_fastq(fq):
            if len(seq) >= stage_cfg['min_read_length']:
                reads.append((name, seq, qual, fq.name))
    reads.sort(key=lambda row: len(row[1]), reverse=True)
    reads = reads[: stage_cfg.get('max_reads', len(reads))]

    aligner = mp.Aligner(str(reference_fa), preset='map-ont')
    fasta = pysam.FastaFile(str(reference_fa))
    header = {'HD': {'VN': '1.6', 'SO': 'unsorted'}, 'SQ': [{'SN': name, 'LN': length} for name, length in zip(fasta.references, fasta.lengths)]}
    ref_to_id = {name: idx for idx, name in enumerate(fasta.references)}

    with pysam.AlignmentFile(str(unsorted_bam), 'wb', header=header) as bam:
        for name, seq, qual, source in reads:
            hits = [hit for hit in aligner.map(seq) if hit.is_primary]
            if not hits:
                continue
            hit = max(hits, key=lambda h: (h.mapq, h.mlen, h.blen))
            aln = pysam.AlignedSegment()
            aln.query_name = f'{source}|{name}'
            aln.flag = 16 if hit.strand < 0 else 0
            aln.reference_id = ref_to_id[hit.ctg]
            aln.reference_start = hit.r_st
            aln.mapping_quality = int(hit.mapq)
            aln.cigarstring = full_cigar(hit, len(seq))
            if hit.strand < 0:
                aln.query_sequence = revcomp(seq)
                aln.query_qualities = pysam.qualitystring_to_array(qual[::-1])
            else:
                aln.query_sequence = seq
                aln.query_qualities = pysam.qualitystring_to_array(qual)
            bam.write(aln)
    pysam.sort('-o', str(sorted_bam), str(unsorted_bam))
    pysam.index(str(sorted_bam))

    build_read_encoding = preprocessing.build_read_encoding
    build_window_example = preprocessing.build_window_example
    iter_windows = preprocessing.iter_windows
    VariantLookup = preprocessing.VariantLookup
    counts = {'train': 0, 'val': 0, 'test': 0}
    files = {'train': open(train_jsonl, 'w', encoding='utf-8'), 'val': open(val_jsonl, 'w', encoding='utf-8'), 'test': open(test_jsonl, 'w', encoding='utf-8')}
    try:
        with pysam.AlignmentFile(str(sorted_bam), 'rb', reference_filename=str(reference_fa)) as bam:
            for aln in bam.fetch(until_eof=True):
                enc = build_read_encoding(aln, fasta, max_insertions_per_pos=2)
                if enc is None or len(enc.target_bases) < stage_cfg['min_window_size']:
                    continue
                digest = int(hashlib.md5(enc.read_id.encode()).hexdigest(), 16) % 10
                split = 'train' if digest < 8 else ('val' if digest == 8 else 'test')
                for ws, we in iter_windows(len(enc.target_bases), stage_cfg['window_size'], stage_cfg['window_overlap'], stage_cfg['min_window_size']):
                    ex = build_window_example(
                        target_aln=aln,
                        read_encoding=enc,
                        support_bam=bam,
                        variant_lookup=VariantLookup(None),
                        phased_variant_lookup=None,
                        confidence_lookup=None,
                        region_lookups={},
                        window_start=ws,
                        window_end=we,
                        max_supports=stage_cfg['max_supports'],
                        min_supports_per_window=stage_cfg['min_supports_per_window'],
                        min_mapq=stage_cfg['min_mapq'],
                        min_confident_fraction=0.0,
                        min_mapped_fraction=0.6,
                        support_disagreement_threshold=0.7,
                        min_support_depth=2,
                        max_insertions_per_pos=2,
                    )
                    if ex is None:
                        continue
                    files[split].write(json.dumps(ex) + '\n')
                    counts[split] += 1
    finally:
        for handle in files.values():
            handle.close()

    manifest = {'counts': counts, 'reference_fasta': str(reference_fa), 'reads_bam': str(sorted_bam), 'haplotagged_bam': str(sorted_bam), 'region_beds': {}, 'train_path': str(train_jsonl), 'val_path': str(val_jsonl), 'test_path': str(test_jsonl)}
    (stage_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
    return manifest

In [8]:
def expand_small_benchmark_coverage_variants(profile):
    if EXPERIMENT_PRESET != 'hg002_chr20_small_benchmark':
        return profile
    expanded = copy.deepcopy(profile)
    base_primary = next((dataset for dataset in expanded['datasets'] if dataset['name'] == 'hg002_chr20_primary'), None)
    if base_primary is None:
        return expanded
    coverage_datasets = []
    for coverage in expanded.get('coverage_targets', []):
        cov_value = int(coverage)
        cov_tag = f'{cov_value}x'
        reads_env = f'HG002_CHR20_{cov_value}X_READS_BAM'
        hap_env = f'HG002_CHR20_{cov_value}X_HAPLOTAGGED_BAM'
        reads_bam = os.environ.get(reads_env, '').strip()
        if not reads_bam:
            default_candidate = ROOT / 'data' / 'subsets' / f'hg002_chr20_{cov_tag}' / 'HG002.giab.subset.bam'
            if default_candidate.exists():
                reads_bam = str(default_candidate.resolve())
            else:
                reads_bam = base_primary.get('reads_bam', '')
        haplotagged_bam = os.environ.get(hap_env, '').strip() or reads_bam
        dataset = copy.deepcopy(base_primary)
        dataset['name'] = f"{base_primary['name']}_{cov_tag}"
        dataset['coverage_tag'] = cov_tag
        dataset['role'] = 'primary' if not coverage_datasets else 'train_variant'
        dataset['reads_bam'] = str(Path(reads_bam).resolve()) if reads_bam else ''
        dataset['haplotagged_bam'] = str(Path(haplotagged_bam).resolve()) if haplotagged_bam else dataset['reads_bam']
        reqs = copy.deepcopy(dataset.get('_requirements', {}))
        if 'reads_bam' in reqs:
            reqs['reads_bam']['env_var'] = reads_env
            reqs['reads_bam']['required'] = True
        if 'haplotagged_bam' in reqs:
            reqs['haplotagged_bam']['env_var'] = hap_env
        dataset['_requirements'] = reqs
        coverage_datasets.append(dataset)
    expanded['datasets'] = coverage_datasets + [dataset for dataset in expanded['datasets'] if dataset['role'] != 'primary']
    return expanded

profile = expand_small_benchmark_coverage_variants(resolve_profile(PROFILE_PRESETS[EXPERIMENT_PRESET], ROOT))
selected_run_set = set(SELECTED_RUNS)
unknown_runs = sorted(selected_run_set - {run['name'] for run in ALL_RUNS})
if unknown_runs:
    raise ValueError(f'Unknown run names requested: {unknown_runs}')
SELECTED_MODEL_RUNS = [run for run in ALL_RUNS if run['name'] in selected_run_set]
if not SELECTED_MODEL_RUNS:
    raise ValueError('SELECTED_RUNS must contain at least one run name.')
def dataset_ready_for_mode(dataset_cfg, mode):
    if mode == 'preprocessed_dataset':
        return all(dataset_cfg.get(field) and Path(dataset_cfg[field]).exists() for field in ['train_path', 'val_path', 'test_path'])
    return bool(dataset_cfg.get('reads_bam')) and Path(dataset_cfg['reads_bam']).exists() and bool(dataset_cfg.get('reference_fasta')) and Path(dataset_cfg['reference_fasta']).exists()

PRIMARY_ANALYSIS_RUN = SELECTED_MODEL_RUNS[0]['name']
TRAIN_DATASETS = [dataset for dataset in profile['datasets'] if dataset['role'] in {'primary', 'train_variant'}]
PRIMARY_DATASET = TRAIN_DATASETS[0]
EVAL_DATASETS = [dataset for dataset in profile['datasets'] if dataset['role'] in {'primary', 'train_variant', 'eval_only'}]
ACTIVE_TRAIN_DATASETS = [dataset for dataset in TRAIN_DATASETS if dataset_ready_for_mode(dataset, profile['mode'])]
ACTIVE_EVAL_DATASETS = [dataset for dataset in EVAL_DATASETS if dataset_ready_for_mode(dataset, profile['mode'])]
SKIPPED_OPTIONAL_DATASETS = [dataset['name'] for dataset in EVAL_DATASETS if dataset not in ACTIVE_EVAL_DATASETS and dataset['role'] == 'eval_only']
RESOLVED_DEVICE, RESOLVED_MIXED_PRECISION = pick_device(PREFERRED_DEVICE)

RUN_ROOT = ROOT / 'data' / profile['run_name']
CKPT_ROOT = ROOT / 'checkpoints' / profile['run_name']
OUT_ROOT = ROOT / 'outputs' / profile['run_name']
CONFIG_ROOT = ROOT / 'configs' / profile['run_name']
PLAN_PATH = OUT_ROOT / 'benchmark_plan.json'
for path in [RUN_ROOT, CKPT_ROOT, OUT_ROOT, CONFIG_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

validation_rows = validate_profile_inputs(profile)
validation_df = pd.DataFrame(validation_rows)
missing_inputs_df = pd.DataFrame(missing_required_inputs(validation_rows))
print('python:', sys.executable)
print('preset:', EXPERIMENT_PRESET)
print('selected runs:', SELECTED_RUNS)
print('device:', RESOLVED_DEVICE, 'mixed_precision:', RESOLVED_MIXED_PRECISION)
print('training datasets:', [dataset['name'] for dataset in TRAIN_DATASETS])
print('active training datasets:', [dataset['name'] for dataset in ACTIVE_TRAIN_DATASETS])
print('active eval datasets:', [dataset['name'] for dataset in ACTIVE_EVAL_DATASETS])
if SKIPPED_OPTIONAL_DATASETS:
    print('skipping optional eval datasets with missing inputs:', SKIPPED_OPTIONAL_DATASETS)
display(validation_df)
display(pd.DataFrame(profile['stages']))
if VALIDATE_ON_PLAN and not missing_inputs_df.empty:
    raise FileNotFoundError('Missing required benchmark inputs:\n' + missing_inputs_df.to_string(index=False))


python: /Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python
preset: hg002_chr20_small_benchmark
selected runs: ['full']
device: mps mixed_precision: False
training datasets: ['hg002_chr20_primary_8x', 'hg002_chr20_primary_12x', 'hg002_chr20_primary_20x']
active training datasets: ['hg002_chr20_primary_8x', 'hg002_chr20_primary_12x', 'hg002_chr20_primary_20x']
active eval datasets: ['hg002_chr20_primary_8x', 'hg002_chr20_primary_12x', 'hg002_chr20_primary_20x']
skipping optional eval datasets with missing inputs: ['hg003_chr20_transfer', 'hg004_chr20_transfer']


,dataset,field,path,exists,required,env_var
0,hg002_chr20_primary_8x,manifest_path,,False,False,
1,hg002_chr20_primary_8x,train_path,,False,False,
2,hg002_chr20_primary_8x,val_path,,False,False,
3,hg002_chr20_primary_8x,test_path,,False,False,
4,hg002_chr20_primary_8x,reference_fasta,/Users/shanejayasundera/LRS-Error-Correction/d...,True,True,GIAB_REFERENCE_FASTA
5,hg002_chr20_primary_8x,reads_bam,/Users/shanejayasundera/LRS-Error-Correction/d...,True,True,HG002_CHR20_8X_READS_BAM
6,hg002_chr20_primary_8x,truth_vcf,/Users/shanejayasundera/LRS-Error-Correction/d...,True,True,HG002_TRUTH_VCF
7,hg002_chr20_primary_8x,confident_bed,/Users/shanejayasundera/LRS-Error-Correction/d...,True,True,HG002_CONFIDENT_BED
8,hg002_chr20_primary_8x,phased_vcf,,False,False,HG002_PHASED_VCF
9,hg002_chr20_primary_8x,haplotagged_bam,/Users/shanejayasundera/LRS-Error-Correction/d...,True,False,HG002_CHR20_8X_HAPLOTAGGED_BAM


,name,window_size,window_overlap,min_window_size,max_target_len,max_support_len,batch_size,epochs,min_read_length,max_supports,min_supports_per_window,min_mapq,max_examples_per_split,early_stopping_patience
0,chr20_small,2048,512,1024,2048,2048,2,7,10000,24,4,20,1500,2


In [9]:
def build_preprocess_command(dataset_cfg, stage_cfg, output_dir):
    cmd = [
        sys.executable,
        'scripts/preprocess_real_data.py',
        '--reads-bam', dataset_cfg['reads_bam'],
        '--reference-fasta', dataset_cfg['reference_fasta'],
        '--output-dir', str(output_dir),
        '--window-size', str(stage_cfg['window_size']),
        '--window-overlap', str(stage_cfg['window_overlap']),
        '--min-window-size', str(stage_cfg['min_window_size']),
        '--max-supports', str(stage_cfg['max_supports']),
        '--min-supports-per-window', str(stage_cfg['min_supports_per_window']),
        '--min-mapq', str(stage_cfg['min_mapq']),
        '--min-read-length', str(stage_cfg['min_read_length']),
        '--max-deletion-length', str(stage_cfg.get('max_deletion_length', 4)),
        '--train-contigs', dataset_cfg['train_contigs'],
        '--val-contigs', dataset_cfg['val_contigs'],
        '--test-contigs', dataset_cfg['test_contigs'],
    ]
    train_set = {token.strip() for token in dataset_cfg['train_contigs'].split(',') if token.strip()}
    val_set = {token.strip() for token in dataset_cfg['val_contigs'].split(',') if token.strip()}
    test_set = {token.strip() for token in dataset_cfg['test_contigs'].split(',') if token.strip()}
    if (train_set & (val_set | test_set)) or (val_set & test_set):
        cmd += ['--allow-shared-holdout-contigs']
    if stage_cfg.get('max_examples_per_split'):
        cmd += ['--max-examples-per-split', str(stage_cfg['max_examples_per_split'])]
    if dataset_cfg.get('truth_vcf'):
        cmd += ['--truth-vcf', dataset_cfg['truth_vcf']]
    if dataset_cfg.get('phased_vcf'):
        cmd += ['--phased-vcf', dataset_cfg['phased_vcf']]
    if dataset_cfg.get('confident_bed'):
        cmd += ['--confident-bed', dataset_cfg['confident_bed']]
    if dataset_cfg.get('haplotagged_bam'):
        cmd += ['--haplotagged-bam', dataset_cfg['haplotagged_bam']]
    for region_name, region_path in profile.get('region_beds', {}).items():
        if region_path:
            cmd += ['--region-bed', f'{region_name}={region_path}']
    return cmd


def build_config(stage_cfg, dataset_dir, checkpoint_dir, manifest_path, region_names, support_mode, init_checkpoint='', checkpoint_weights=None):
    checkpoint_weights = checkpoint_weights or profile['checkpoint_weights']
    effective_batch_size = 1 if RESOLVED_DEVICE == 'mps' else stage_cfg['batch_size']
    return {
        'model': {
            'd_model': 64,
            'num_heads': 4,
            'num_layers': 2,
            'ff_mult': 2,
            'dropout': 0.1,
            'target_feature_dim': 10,
            'support_feature_dim': 17,
            'max_supports': stage_cfg['max_supports'],
            'max_insertions_per_pos': 2,
            'max_deletion_length': stage_cfg.get('max_deletion_length', 4),
            'run_length_classes': 8,
            'conv_kernel_size': 5,
            'support_mode': support_mode,
            'apply_hard_edit_support_filter': True,
            'hard_edit_min_support_agreement': 0.95,
            'hard_edit_max_support_entropy': 0.15,
            'hard_edit_min_support_depth': max(2.0, float(stage_cfg['min_supports_per_window'])),
            'hard_edit_filter_logit_bias': 6.0,
            'inference_hard_edit_confidence_threshold': 0.9,
            'inference_hard_edit_temperature': 0.75,
            'inference_sub_confidence_threshold': 0.8,
            'inference_del_confidence_threshold': 0.93,
            'inference_ins_confidence_threshold': 0.75,
            'deletion_commit_trust_threshold': 0.85,
            'deletion_candidate_threshold': 0.85,
            'inference_use_deletion_consistency_check': True,
        },
        'data': {
            'train_path': safe_rel(dataset_dir / 'train.jsonl'),
            'val_path': safe_rel(dataset_dir / 'val.jsonl'),
            'test_path': safe_rel(dataset_dir / 'test.jsonl'),
            'manifest_path': safe_rel(manifest_path),
            'max_target_len': stage_cfg['max_target_len'],
            'max_supports': stage_cfg['max_supports'],
            'max_support_len': stage_cfg['max_support_len'],
            'num_workers': 0,
            'deletion_oversample_weight': 2.5,
            'region_names': sorted(region_names),
        },
        'train': {
            'seed': 42,
            'batch_size': effective_batch_size,
            'epochs': stage_cfg['epochs'],
            'lr': 1e-4,
            'weight_decay': 0.01,
            'grad_clip_norm': 0.5,
            'device': RESOLVED_DEVICE,
            'mixed_precision': RESOLVED_MIXED_PRECISION,
            'log_every': 20,
            'eval_every': 1,
            'save_dir': safe_rel(checkpoint_dir),
            'init_checkpoint': init_checkpoint,
            'checkpoint_metric': 'composite_sequence',
            'checkpoint_metric_mode': 'max',
            'checkpoint_overcorrection_weight': checkpoint_weights.get('overcorrection', 0.0),
            'checkpoint_hard_edit_fp_weight': checkpoint_weights.get('hard_edit_fp', 0.0),
            'checkpoint_length_ratio_weight': checkpoint_weights.get('length_ratio', 0.0),
            'early_stopping_patience': stage_cfg.get('early_stopping_patience', 0),
            'oversample_deletion_windows': True,
        },
        'loss': {
            'lambda_edit': 1.0,
            'lambda_sequence': 0.5,
            'lambda_length': 0.3,
            'lambda_insertion_count': 0.2,
            'lambda_trust': 0.1,
            'lambda_hard_edit': 0.4,
            'lambda_hard_edit_precision': 0.8,
            'lambda_selective_hard_edit': 0.2,
            'lambda_deletion_candidate': 0.4,
            'lambda_deletion_length': 0.2,
            'lambda_run_length_aux': 0.1,
            'lambda_deletion_positive_reward': 0.15,
            'lambda_support': 0.0,
            'lambda_preserve': 0.2,
            'lambda_uncertainty': 0.1,
            'homopolymer_weight_scale': 0.15,
            'label_smoothing': 0.0,
            'edit_class_weights': [],
            'auto_edit_class_weights': True,
            'edit_class_weight_power': 1.0,
            'edit_class_weight_count_smoothing': 1.0,
            'edit_class_weight_min': 0.25,
            'edit_class_weight_max': 8.0,
            'substitution_loss_scale': 1.75,
            'deletion_loss_scale': 1.25,
            'insertion_loss_scale': 1.1,
            'hard_edit_uncertainty_power': 2.0,
            'hard_edit_entropy_threshold': 0.15,
            'hard_edit_agreement_threshold': 0.95,
            'hard_edit_entropy_scale': 2.5,
            'hard_edit_low_agreement_scale': 3.5,
            'hard_edit_false_positive_weight': 8.0,
            'hard_edit_false_negative_weight': 1.0,
            'selective_hard_edit_confidence_threshold': 0.6,
            'selective_hard_edit_uncertainty_threshold': 0.4,
            'selective_hard_edit_min_support_agreement': 0.93,
            'deletion_focal_gamma': 2.0,
            'deletion_false_positive_weight': 8.0,
            'deletion_false_negative_weight': 1.0,
            'deletion_positive_reward_scale': 1.5,
        },
    }


def selection_weights_for_profile(profile):
    weights = dict(profile.get('checkpoint_weights', {}))
    if EXPERIMENT_PRESET == 'hg002_chr20_small_benchmark':
        weights['hard_edit_fp'] = 0.0
        weights['length_ratio'] = 0.0
    return weights


def eval_targets_for_train_dataset(train_dataset_cfg):
    if EXPERIMENT_PRESET == 'hg002_chr20_small_benchmark':
        return [train_dataset_cfg]
    if train_dataset_cfg['name'] == PRIMARY_DATASET['name']:
        return EVAL_DATASETS
    return [train_dataset_cfg]


def baseline_targets_for_train_dataset(train_dataset_cfg):
    if EXPERIMENT_PRESET == 'hg002_chr20_small_benchmark':
        return [train_dataset_cfg]
    return eval_targets_for_train_dataset(train_dataset_cfg)


preprocess_plans = []
train_plans = []
eval_plans = []
baseline_plans = []
external_baseline_plans = []
downstream_plans = []
selection_weights = selection_weights_for_profile(profile)

for stage_idx, stage_cfg in enumerate(profile['stages']):
    stage_name = stage_cfg['name']
    previous_stage_name = profile['stages'][stage_idx - 1]['name'] if stage_idx > 0 else None
    stage_dirs = {}
    stage_manifest_paths = {}

    for dataset_cfg in ACTIVE_EVAL_DATASETS:
        if profile['mode'] == 'preprocessed_dataset':
            dataset_stage_dir = Path(dataset_cfg['train_path']).resolve().parent
            manifest_path = Path(dataset_cfg['manifest_path']).resolve()
        else:
            dataset_stage_dir = RUN_ROOT / dataset_cfg['name'] / stage_name
            dataset_stage_dir.mkdir(parents=True, exist_ok=True)
            manifest_path = dataset_stage_dir / 'manifest.json'
            preprocess_plans.append({
                'stage': stage_name,
                'dataset': dataset_cfg['name'],
                'coverage': dataset_cfg.get('coverage_tag', ''),
                'role': dataset_cfg['role'],
                'manifest_path': str(manifest_path),
                'cmd': build_preprocess_command(dataset_cfg, stage_cfg, dataset_stage_dir),
            })
        stage_dirs[dataset_cfg['name']] = dataset_stage_dir
        stage_manifest_paths[dataset_cfg['name']] = manifest_path

    region_names = set(profile.get('region_beds', {}).keys()) | {'homopolymer', 'variant_rich', 'tandem_repeat'}

    for train_dataset_cfg in ACTIVE_TRAIN_DATASETS:
        coverage_tag = train_dataset_cfg.get('coverage_tag', train_dataset_cfg['name'])
        train_stage_dir = stage_dirs[train_dataset_cfg['name']]
        eval_targets = eval_targets_for_train_dataset(train_dataset_cfg)
        baseline_targets = baseline_targets_for_train_dataset(train_dataset_cfg)
        stage_output_dir = OUT_ROOT / coverage_tag / stage_name
        stage_output_dir.mkdir(parents=True, exist_ok=True)

        for run in SELECTED_MODEL_RUNS:
            checkpoint_dir = CKPT_ROOT / coverage_tag / stage_name / run['name']
            checkpoint_dir.mkdir(parents=True, exist_ok=True)
            config_path = CONFIG_ROOT / f'{coverage_tag}_{stage_name}_{run["name"]}.yaml'
            init_checkpoint = ''
            if previous_stage_name is not None:
                init_checkpoint = safe_rel(CKPT_ROOT / coverage_tag / previous_stage_name / run['name'] / 'best_model.pt')
            cfg = build_config(
                stage_cfg,
                train_stage_dir,
                checkpoint_dir,
                stage_manifest_paths[train_dataset_cfg['name']],
                region_names,
                support_mode=run['support_mode'],
                init_checkpoint=init_checkpoint,
                checkpoint_weights=selection_weights,
            )
            with open(config_path, 'w', encoding='utf-8') as handle:
                yaml.safe_dump(cfg, handle, sort_keys=False)
            train_plans.append({
                'coverage': coverage_tag,
                'stage': stage_name,
                'train_dataset': train_dataset_cfg['name'],
                'run': run['name'],
                'config_path': str(config_path),
                'checkpoint_dir': str(checkpoint_dir),
                'cmd': [sys.executable, 'scripts/train.py', '--config', str(config_path)],
            })

            if stage_idx == len(profile['stages']) - 1:
                for dataset_cfg in eval_targets:
                    dataset_stage_dir = stage_dirs[dataset_cfg['name']]
                    dataset_output_dir = stage_output_dir / dataset_cfg['name'] / run['name']
                    dataset_output_dir.mkdir(parents=True, exist_ok=True)
                    eval_plans.append({
                        'coverage': coverage_tag,
                        'train_dataset': train_dataset_cfg['name'],
                        'stage': stage_name,
                        'dataset': dataset_cfg['name'],
                        'run': run['name'],
                        'summary_path': str(dataset_output_dir / 'test_summary.json'),
                        'predictions_path': str(dataset_output_dir / 'test_predictions.jsonl'),
                        'cmd': [
                            sys.executable, 'scripts/test.py', '--config', str(config_path),
                            '--checkpoint', str(checkpoint_dir / 'best_model.pt'),
                            '--data-path', str(Path(dataset_cfg['test_path']).resolve() if profile['mode'] == 'preprocessed_dataset' else dataset_stage_dir / 'test.jsonl'),
                            '--summary-out', str(dataset_output_dir / 'test_summary.json'),
                            '--predictions-out', str(dataset_output_dir / 'test_predictions.jsonl'),
                        ],
                    })

                if run['name'] == PRIMARY_ANALYSIS_RUN:
                    for dataset_cfg in baseline_targets:
                        dataset_stage_dir = stage_dirs[dataset_cfg['name']]
                        baseline_out = stage_output_dir / dataset_cfg['name'] / 'baselines'
                        baseline_out.mkdir(parents=True, exist_ok=True)
                        baseline_plans.extend([
                            {
                                'kind': 'internal',
                                'coverage': coverage_tag,
                                'train_dataset': train_dataset_cfg['name'],
                                'baseline': 'no_edit',
                                'dataset': dataset_cfg['name'],
                                'summary_path': str(baseline_out / 'no_edit_summary.json'),
                                'cmd': [sys.executable, 'scripts/no_edit_baseline.py', '--config', str(config_path), '--data-path', str(Path(dataset_cfg['test_path']).resolve() if profile['mode'] == 'preprocessed_dataset' else dataset_stage_dir / 'test.jsonl'), '--summary-out', str(baseline_out / 'no_edit_summary.json'), '--predictions-out', str(baseline_out / 'no_edit_predictions.jsonl')],
                            },
                            {
                                'kind': 'internal',
                                'coverage': coverage_tag,
                                'train_dataset': train_dataset_cfg['name'],
                                'baseline': 'consensus',
                                'dataset': dataset_cfg['name'],
                                'summary_path': str(baseline_out / 'consensus_summary.json'),
                                'cmd': [sys.executable, 'scripts/consensus_baseline.py', '--config', str(config_path), '--data-path', str(Path(dataset_cfg['test_path']).resolve() if profile['mode'] == 'preprocessed_dataset' else dataset_stage_dir / 'test.jsonl'), '--summary-out', str(baseline_out / 'consensus_summary.json'), '--predictions-out', str(baseline_out / 'consensus_predictions.jsonl')],
                            },
                        ])

                    if train_dataset_cfg['name'] == PRIMARY_DATASET['name']:
                        for dataset_cfg in eval_targets:
                            external_out = stage_output_dir / dataset_cfg['name'] / 'external_baselines'
                            external_out.mkdir(parents=True, exist_ok=True)
                            for tool in ['herro', 'dechat']:
                                external_baseline_plans.append({
                                    'coverage': coverage_tag,
                                    'tool': tool,
                                    'dataset': dataset_cfg['name'],
                                    'summary_path': str(external_out / f'{tool}_summary.json'),
                                    'cmd': [sys.executable, 'scripts/run_external_baseline.py', '--tool', tool, '--reads-bam', dataset_cfg['reads_bam'], '--reference-fasta', dataset_cfg['reference_fasta'], '--output-dir', str(external_out / tool), '--summary-out', str(external_out / f'{tool}_summary.json')],
                                })

                        for dataset_cfg in eval_targets:
                            eval_row = next(row for row in eval_plans if row['coverage'] == coverage_tag and row['dataset'] == dataset_cfg['name'] and row['run'] == PRIMARY_ANALYSIS_RUN)
                            downstream_out = stage_output_dir / dataset_cfg['name'] / 'downstream'
                            downstream_out.mkdir(parents=True, exist_ok=True)
                            corrected_fasta = downstream_out / 'corrected_reads.fasta'
                            downstream_plans.extend([
                                {
                                    'kind': 'export',
                                    'coverage': coverage_tag,
                                    'dataset': dataset_cfg['name'],
                                    'summary_path': str(downstream_out / 'export_summary.json'),
                                    'cmd': [sys.executable, 'scripts/export_predictions_to_fastx.py', '--predictions-jsonl', eval_row['predictions_path'], '--output', str(corrected_fasta), '--format', 'fasta'],
                                },
                                {
                                    'kind': 'assembly',
                                    'coverage': coverage_tag,
                                    'dataset': dataset_cfg['name'],
                                    'summary_path': str(downstream_out / 'assembly_summary.json'),
                                    'cmd': [sys.executable, 'scripts/run_assembly_eval.py', '--reads-fastx', str(corrected_fasta), '--reference-fasta', dataset_cfg['reference_fasta'], '--output-dir', str(downstream_out / 'assembly'), '--summary-out', str(downstream_out / 'assembly_summary.json')],
                                },
                                {
                                    'kind': 'variant_eval',
                                    'coverage': coverage_tag,
                                    'dataset': dataset_cfg['name'],
                                    'summary_path': str(downstream_out / 'variant_eval_summary.json'),
                                    'cmd': [sys.executable, 'scripts/run_variant_eval.py', '--reads-fastx', str(corrected_fasta), '--reference-fasta', dataset_cfg['reference_fasta'], '--truth-vcf', dataset_cfg['truth_vcf'], '--confident-bed', dataset_cfg['confident_bed'], '--output-dir', str(downstream_out / 'variant_eval'), '--summary-out', str(downstream_out / 'variant_eval_summary.json')],
                                },
                            ])

benchmark_plan = {
    'preset': EXPERIMENT_PRESET,
    'selected_runs': SELECTED_RUNS,
    'device': RESOLVED_DEVICE,
    'mixed_precision': RESOLVED_MIXED_PRECISION,
    'selection_weights': selection_weights,
    'profile': profile,
    'preprocess_plans': preprocess_plans,
    'train_plans': train_plans,
    'eval_plans': eval_plans,
    'baseline_plans': baseline_plans,
    'external_baseline_plans': external_baseline_plans,
    'downstream_plans': downstream_plans,
}
PLAN_PATH.write_text(json.dumps(benchmark_plan, indent=2))
display(pd.DataFrame(train_plans))
print('plan written to', PLAN_PATH)

,coverage,stage,train_dataset,run,config_path,checkpoint_dir,cmd
0,8x,chr20_small,hg002_chr20_primary_8x,full,/Users/shanejayasundera/LRS-Error-Correction/c...,/Users/shanejayasundera/LRS-Error-Correction/c...,[/Users/shanejayasundera/anaconda3/envs/lrs_er...
1,12x,chr20_small,hg002_chr20_primary_12x,full,/Users/shanejayasundera/LRS-Error-Correction/c...,/Users/shanejayasundera/LRS-Error-Correction/c...,[/Users/shanejayasundera/anaconda3/envs/lrs_er...
2,20x,chr20_small,hg002_chr20_primary_20x,full,/Users/shanejayasundera/LRS-Error-Correction/c...,/Users/shanejayasundera/LRS-Error-Correction/c...,[/Users/shanejayasundera/anaconda3/envs/lrs_er...


plan written to /Users/shanejayasundera/LRS-Error-Correction/outputs/hg002_chr20_small_benchmark/benchmark_plan.json


In [10]:
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT / 'src')
env['PYTHONUNBUFFERED'] = '1'
env.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
FORCE_PREPROCESS = os.environ.get('OMEGA_FORCE_PREPROCESS', '0').strip() in {'1', 'true', 'TRUE', 'yes', 'YES'}
AUTO_PREPROCESS = profile['mode'] != 'preprocessed_dataset' and any([RUN_TRAIN, RUN_TEST, RUN_BASELINES, RUN_EXTERNAL_BASELINES, RUN_DOWNSTREAM])
EFFECTIVE_RUN_PREPROCESS = RUN_PREPROCESS or AUTO_PREPROCESS or FORCE_PREPROCESS
if AUTO_PREPROCESS and not RUN_PREPROCESS:
    print('auto-enabling preprocess because this preset requires derived JSONL datasets before train/test/baselines')
if FORCE_PREPROCESS:
    print('forcing preprocess because OMEGA_FORCE_PREPROCESS is enabled')
missing_required = missing_required_inputs(validation_rows)
if any([EFFECTIVE_RUN_PREPROCESS, RUN_TRAIN, RUN_TEST, RUN_BASELINES, RUN_EXTERNAL_BASELINES, RUN_DOWNSTREAM]) and missing_required:
    raise FileNotFoundError('Cannot execute benchmark because required inputs are missing:\n' + pd.DataFrame(missing_required).to_string(index=False))

def preprocess_outputs_ready(row):
    manifest_path = Path(row['manifest_path'])
    output_dir = manifest_path.parent
    required = [manifest_path, output_dir / 'train.jsonl', output_dir / 'val.jsonl', output_dir / 'test.jsonl']
    return all(path.exists() and path.stat().st_size > 0 for path in required)

if EFFECTIVE_RUN_PREPROCESS:
    if profile['mode'] == 'local_fastq_demo':
        for stage_cfg in profile['stages']:
            stage_dir = RUN_ROOT / PRIMARY_DATASET['name'] / stage_cfg['name']
            manifest = prepare_demo_stage(profile, stage_cfg, stage_dir)
            print(json.dumps(manifest, indent=2))
    else:
        for row in preprocess_plans:
            if not FORCE_PREPROCESS and preprocess_outputs_ready(row):
                print(f'\n=== preprocess {row["stage"]} / {row["dataset"]} ===')
                print(f'skipping preprocess; found existing dataset at {Path(row["manifest_path"]).parent}')
                continue
            print(f'\n=== preprocess {row["stage"]} / {row["dataset"]} ===')
            run_cmd(row['cmd'], env=env)

if RUN_TRAIN:
    for row in train_plans:
        print(f'\n=== train {row["stage"]} / {row["run"]} ===')
        run_cmd(row['cmd'], env=env)

if RUN_TEST:
    for row in eval_plans:
        print(f'\n=== test {row["dataset"]} / {row["run"]} ===')
        run_cmd(row['cmd'], env=env)

if RUN_BASELINES:
    for row in baseline_plans:
        print(f'\n=== baseline {row["baseline"]} / {row["dataset"]} ===')
        run_cmd(row['cmd'], env=env)

if RUN_EXTERNAL_BASELINES:
    for row in external_baseline_plans:
        print(f'\n=== external {row["tool"]} / {row["dataset"]} ===')
        run_cmd(row['cmd'], env=env)

if RUN_DOWNSTREAM:
    for row in downstream_plans:
        print(f'\n=== downstream {row["kind"]} / {row["dataset"]} ===')
        run_cmd(row['cmd'], env=env)

auto-enabling preprocess because this preset requires derived JSONL datasets before train/test/baselines

=== preprocess chr20_small / hg002_chr20_primary_8x ===
skipping preprocess; found existing dataset at /Users/shanejayasundera/LRS-Error-Correction/data/hg002_chr20_small_benchmark/hg002_chr20_primary_8x/chr20_small

=== preprocess chr20_small / hg002_chr20_primary_12x ===
skipping preprocess; found existing dataset at /Users/shanejayasundera/LRS-Error-Correction/data/hg002_chr20_small_benchmark/hg002_chr20_primary_12x/chr20_small

=== preprocess chr20_small / hg002_chr20_primary_20x ===
skipping preprocess; found existing dataset at /Users/shanejayasundera/LRS-Error-Correction/data/hg002_chr20_small_benchmark/hg002_chr20_primary_20x/chr20_small

=== train chr20_small / full ===
/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/train.py --config /Users/shanejayasundera/LRS-Error-Correction/configs/hg002_chr20_small_benchmark/8x_chr20_small_full.yaml


python(74471) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/Users/shanejayasundera/LRS-Error-Correction/scripts/train.py:249: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg.train.mixed_precision and device.type == "cuda")


resolved edit class weights:
[
  {
    "token_id": 0,
    "token": "COPY",
    "weight": 0.25
  },
  {
    "token_id": 1,
    "token": "SUB_A",
    "weight": 8.0
  },
  {
    "token_id": 2,
    "token": "SUB_C",
    "weight": 8.0
  },
  {
    "token_id": 3,
    "token": "SUB_G",
    "weight": 8.0
  },
  {
    "token_id": 4,
    "token": "SUB_T",
    "weight": 8.0
  },
  {
    "token_id": 5,
    "token": "DEL",
    "weight": 3.66989803314209
  },
  {
    "token_id": 6,
    "token": "INS_A",
    "weight": 8.0
  },
  {
    "token_id": 7,
    "token": "INS_C",
    "weight": 8.0
  },
  {
    "token_id": 8,
    "token": "INS_G",
    "weight": 8.0
  },
  {
    "token_id": 9,
    "token": "INS_T",
    "weight": 8.0
  },
  {
    "token_id": 10,
    "token": "PAD",
    "weight": 0.0
  }
]
epoch 1/7


/Users/shanejayasundera/LRS-Error-Correction/scripts/train.py:131: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision and device.type == "cuda"):


train step=20: {"loss": 7.800259399414062, "edit_loss": 1.821464255452156, "sequence_loss": 1.3686364889144897, "length_loss": 0.194737658649683, "insertion_count_loss": 0.7565184444189071, "trust_loss": 0.5522696852684021, "hard_edit_loss": 0.0017169253508654946, "hard_edit_precision_loss": 4.080880558490753, "selective_hard_edit_loss": 2.4879973888397218, "deletion_candidate_loss": 0.8193001419305801, "deletion_length_loss": 1.691331398487091, "deletion_positive_reward_loss": 0.7740353375673295, "run_length_loss": 2.100632631778717, "hard_edit_entropy_excess_mean": 0.008567249565385283, "hard_edit_agreement_deficit_mean": 0.011328486143611372, "support_loss": 2.246703690290451, "preserve_loss": 0.8287394523620606, "uncertainty_loss": 1.0863134801387786, "predicted_length": 2309.8701721191405, "target_length": 1940.45, "predicted_insertions": 1474.1000732421876, "target_insertions": 8.0, "trust_gate_mean": 0.5967218190431595, "support_confidence_mean": 0.9082088112831116, "support_unc

...

{
  "train_loss": 3.4953917860984802,
  "train_edit_loss": 1.589380303050081,
  "train_sequence_loss": 0.2983479175517956,
  "train_length_loss": 0.05063475066989215,
  "train_insertion_count_loss": 0.09440870002657176,
  "train_trust_loss": 0.29777673390259346,
  "train_hard_edit_loss": 0.00036417852902419174,
  "train_hard_edit_precision_loss": 0.7333607100894054,
  "train_selective_hard_edit_loss": 3.102954007466634,
  "train_deletion_candidate_loss": 0.07962186723171423,
  "train_deletion_length_loss": 0.9539771243184805,
  "train_deletion_positive_reward_loss": 0.9741135292053222,
  "train_run_length_loss": 0.849033486088117,
  "train_hard_edit_entropy_excess_mean": 0.007181902452740663,
  "train_hard_edit_agreement_deficit_mean": 0.010602421675265455,
  "train_support_loss": 0.2826894157504042,
  "train_preserve_loss": 0.12108392781531438,
  "train_uncertainty_loss": 0.07681295359445114,
  "train_predicted_length": 2044.3794286295572,
  "train_target_length": 1978.888,
  "train_p

python(64243) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/Users/shanejayasundera/LRS-Error-Correction/scripts/train.py:249: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg.train.mixed_precision and device.type == "cuda")


resolved edit class weights:
[
  {
    "token_id": 0,
    "token": "COPY",
    "weight": 0.25
  },
  {
    "token_id": 1,
    "token": "SUB_A",
    "weight": 8.0
  },
  {
    "token_id": 2,
    "token": "SUB_C",
    "weight": 8.0
  },
  {
    "token_id": 3,
    "token": "SUB_G",
    "weight": 8.0
  },
  {
    "token_id": 4,
    "token": "SUB_T",
    "weight": 8.0
  },
  {
    "token_id": 5,
    "token": "DEL",
    "weight": 3.090029239654541
  },
  {
    "token_id": 6,
    "token": "INS_A",
    "weight": 8.0
  },
  {
    "token_id": 7,
    "token": "INS_C",
    "weight": 8.0
  },
  {
    "token_id": 8,
    "token": "INS_G",
    "weight": 8.0
  },
  {
    "token_id": 9,
    "token": "INS_T",
    "weight": 8.0
  },
  {
    "token_id": 10,
    "token": "PAD",
    "weight": 0.0
  }
]
epoch 1/7


/Users/shanejayasundera/LRS-Error-Correction/scripts/train.py:131: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.train.mixed_precision and device.type == "cuda"):


CalledProcessError: Command '['/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python', 'scripts/train.py', '--config', '/Users/shanejayasundera/LRS-Error-Correction/configs/hg002_chr20_small_benchmark/20x_chr20_small_full.yaml']' died with <Signals.SIGKILL: 9>.

In [11]:
def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text())


def usable_score(summary):
    identity = float(summary.get('sequence_identity', 0.0))
    overcorrection = float(summary.get('overcorrection_rate', 0.0))
    return identity - selection_weights.get('overcorrection', 0.0) * overcorrection


correction_rows = []
for row in eval_plans:
    summary = load_json(row['summary_path'])
    if summary is None:
        continue
    correction_rows.append({
        'kind': 'model', 'coverage': row.get('coverage', ''), 'train_dataset': row.get('train_dataset', row['dataset']), 'dataset': row['dataset'], 'name': row['run'],
        'sequence_identity': float(summary.get('sequence_identity', 0.0)),
        'normalized_edit_distance': float(summary.get('sequence_normalized_edit_distance', 0.0)),
        'overcorrection_rate': float(summary.get('overcorrection_rate', 0.0)),
        'hard_edit_false_positive_rate': float(summary.get('hard_edit_false_positive_rate', 0.0)),
        'repeat_sequence_identity': float(summary.get('repeat_sequence_identity', summary.get('repeat_rich_sequence_identity', 0.0))),
        'segmental_duplication_sequence_identity': float(summary.get('segmental_duplication_sequence_identity', 0.0)),
        'variant_rich_sequence_identity': float(summary.get('variant_rich_sequence_identity', 0.0)),
        'homopolymer_core_error_rate': float(summary.get('homopolymer_core_error_rate', 0.0)),
        'phased_variant_core_error_rate': float(summary.get('phased_variant_core_error_rate', 0.0)),
        'usable_correction_score': usable_score(summary),
        'summary_path': row['summary_path'],
    })
for row in baseline_plans:
    summary = load_json(row['summary_path'])
    if summary is None:
        continue
    correction_rows.append({
        'kind': f'baseline:{row["baseline"]}', 'coverage': row.get('coverage', ''), 'train_dataset': row.get('train_dataset', row['dataset']), 'dataset': row['dataset'], 'name': row['baseline'],
        'sequence_identity': float(summary.get('sequence_identity', 0.0)),
        'normalized_edit_distance': float(summary.get('sequence_normalized_edit_distance', 0.0)),
        'overcorrection_rate': float(summary.get('overcorrection_rate', 0.0)),
        'hard_edit_false_positive_rate': float(summary.get('hard_edit_false_positive_rate', 0.0)),
        'repeat_sequence_identity': float(summary.get('repeat_sequence_identity', summary.get('repeat_rich_sequence_identity', 0.0))),
        'segmental_duplication_sequence_identity': float(summary.get('segmental_duplication_sequence_identity', 0.0)),
        'variant_rich_sequence_identity': float(summary.get('variant_rich_sequence_identity', 0.0)),
        'homopolymer_core_error_rate': float(summary.get('homopolymer_core_error_rate', 0.0)),
        'phased_variant_core_error_rate': float(summary.get('phased_variant_core_error_rate', 0.0)),
        'usable_correction_score': usable_score(summary),
        'summary_path': row['summary_path'],
    })
correction_df = pd.DataFrame(correction_rows)
if not correction_df.empty:
    consensus_lookup = correction_df[correction_df['kind'] == 'baseline:consensus'][['coverage', 'dataset', 'sequence_identity', 'normalized_edit_distance', 'overcorrection_rate', 'usable_correction_score']].rename(columns={
        'sequence_identity': 'consensus_sequence_identity',
        'normalized_edit_distance': 'consensus_normalized_edit_distance',
        'overcorrection_rate': 'consensus_overcorrection_rate',
        'usable_correction_score': 'consensus_usable_correction_score',
    })
    correction_df = correction_df.merge(consensus_lookup, on=['coverage', 'dataset'], how='left')
    correction_df['delta_identity_vs_consensus'] = correction_df['sequence_identity'] - correction_df['consensus_sequence_identity'].fillna(0.0)
    correction_df['delta_ned_vs_consensus'] = correction_df['normalized_edit_distance'] - correction_df['consensus_normalized_edit_distance'].fillna(0.0)
    correction_df['delta_score_vs_consensus'] = correction_df['usable_correction_score'] - correction_df['consensus_usable_correction_score'].fillna(0.0)
    display(correction_df.sort_values(['coverage', 'dataset', 'usable_correction_score'], ascending=[True, True, False]).reset_index(drop=True))
else:
    print('No correction summaries found yet.')

external_df = pd.DataFrame([load_json(row['summary_path']) | {'dataset': row['dataset'], 'tool': row['tool']} for row in external_baseline_plans if load_json(row['summary_path']) is not None])
if not external_df.empty:
    display(external_df)
else:
    print('No external baseline summaries found yet.')

assembly_df = pd.DataFrame([load_json(row['summary_path']) | {'dataset': row['dataset']} for row in downstream_plans if row['kind'] == 'assembly' and load_json(row['summary_path']) is not None])
if not assembly_df.empty:
    display(assembly_df)
else:
    print('No assembly summaries found yet.')

variant_df = pd.DataFrame([load_json(row['summary_path']) | {'dataset': row['dataset']} for row in downstream_plans if row['kind'] == 'variant_eval' and load_json(row['summary_path']) is not None])
if not variant_df.empty:
    display(variant_df)
else:
    print('No variant-calling summaries found yet.')

combined_summary = {
    'correction': correction_rows,
    'external_baselines': external_df.to_dict(orient='records') if not external_df.empty else [],
    'assembly': assembly_df.to_dict(orient='records') if not assembly_df.empty else [],
    'variant_calling': variant_df.to_dict(orient='records') if not variant_df.empty else [],
}
summary_path = OUT_ROOT / 'benchmark_summary.json'
summary_path.write_text(json.dumps(combined_summary, indent=2))
print('benchmark summary saved to', summary_path)

No correction summaries found yet.
No external baseline summaries found yet.
No assembly summaries found yet.
No variant-calling summaries found yet.
benchmark summary saved to /Users/shanejayasundera/LRS-Error-Correction/outputs/hg002_chr20_small_benchmark/benchmark_summary.json


In [12]:
print('\n=== preprocess plan ===')
for row in preprocess_plans:
    print(f'\n[{row["stage"]}] [{row["dataset"]}]')
    print(stringify_cmd(row['cmd']))

print('\n=== training plan ===')
for row in train_plans:
    print(f'\n[{row["stage"]}] [{row["run"]}]')
    print('train:', stringify_cmd(row['cmd']))

print('\n=== evaluation plan ===')
for row in eval_plans:
    print(f'\n[{row["dataset"]}] [{row["run"]}]')
    print('test:', stringify_cmd(row['cmd']))

print('\n=== internal baselines ===')
for row in baseline_plans:
    print(f'{row["dataset"]} / {row["baseline"]}:', stringify_cmd(row['cmd']))

print('\n=== external baseline hooks ===')
for row in external_baseline_plans:
    print(f'{row["dataset"]} / {row["tool"]}:', stringify_cmd(row['cmd']))

print('\n=== downstream hooks ===')
for row in downstream_plans:
    print(f'{row["dataset"]} / {row["kind"]}:', stringify_cmd(row['cmd']))


=== preprocess plan ===

[chr20_small] [hg002_chr20_primary_8x]
/Users/shanejayasundera/anaconda3/envs/lrs_err_correct_env/bin/python scripts/preprocess_real_data.py --reads-bam /Users/shanejayasundera/LRS-Error-Correction/data/subsets/hg002_chr20_8x/HG002.giab.subset.bam --reference-fasta /Users/shanejayasundera/LRS-Error-Correction/data/raw/giab/hg002/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna --output-dir /Users/shanejayasundera/LRS-Error-Correction/data/hg002_chr20_small_benchmark/hg002_chr20_primary_8x/chr20_small --window-size 2048 --window-overlap 512 --min-window-size 1024 --max-supports 24 --min-supports-per-window 4 --min-mapq 20 --min-read-length 10000 --max-deletion-length 4 --train-contigs chr20 --val-contigs chr20 --test-contigs chr20 --allow-shared-holdout-contigs --max-examples-per-split 1500 --truth-vcf /Users/shanejayasundera/LRS-Error-Correction/data/raw/giab/hg002/HG002_GRCh38_1_22_v4.2.1_benchmark.vcf.gz --confident-bed /Users/shanejayasundera/LRS-Error-Corre